# Lesson 3, Exercise 1: Post-Training Quantization - Beyond Memory: Speed and Quality Trade-offs

**Goal:**
The primary goal of this exercise is to move beyond simply observing memory reduction from quantization and to comprehensively evaluate its impact. You will quantify and analyze the trade-offs between model memory footprint, inference speed (latency), and the subjective quality of generated text when applying different levels of Post-Training Quantization (PTQ) to a GPT-2 model.

## 2. Imports and Configuration

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import time
import pandas as pd

MODEL_NAME = "gpt2" # Standard GPT-2
PROMPTS = [
    "The capital of France is",
    "Once upon a time, in a land far, far away,",
    "To be or not to be, that is the"
]
MAX_NEW_TOKENS = 50
NUM_TIMING_RUNS = 3 # Number of times to run generation for averaging latency

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 3. Helper Functions

In [8]:
def get_model_memory_footprint(model):
    """Gets model memory footprint in MB."""
    mem_params = sum([param.numel() * param.element_size() for param in model.parameters()]) # TODO: extract memory size for each parameter. Ref: https://discuss.pytorch.org/t/finding-model-size/130275
    mem_bufs = sum([buf.numel() * buf.element_size() for buf in model.buffers()]) # TODO: extract memory size for each buffer. Ref: https://discuss.pytorch.org/t/finding-model-size/130275
    mem = mem_params + mem_bufs # in bytes
    return mem / 1024**2 # convert to MB

def generate_text_and_time(model, tokenizer, prompt, max_new_tokens):
    """Generates text and returns the generated text and latency."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    start_time = time.perf_counter() # Use perf_counter for more precise timing
    if model.device.type == 'cuda':
        torch.cuda.synchronize() # Ensure previous CUDA ops are done
        
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens) # TODO: write generation logic
    
    if model.device.type == 'cuda':
        torch.cuda.synchronize() # Ensure generation is done
    end_time = time.perf_counter()
    
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True) # TODO: extract generated text from output. HINT: use the tokenizer 
    latency = end_time - start_time
    return generated_text, latency

## 4. Main Experiment Logic

In [9]:
results_list = [] # Use a different name to avoid conflict if re-running cells

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Define precision configurations to test
configurations = [
        {"name": "FP16 (Baseline GPU)" if device.type == "cuda" else "FP32 (Baseline CPU)", "load_args": {"torch_dtype": torch.float16 if device.type == "cuda" else torch.float32}},
        {"name": "INT8 (bitsandbytes)", "load_args": {"load_in_8bit": True, "device_map": "auto" if device.type == "cuda" else None}},
        {"name": "NF4 (NormalFloat4)", "load_args": {"load_in_4bit": True, "device_map": "auto"}},
        {"name": "FP4 (Optional Bonus)", "load_args": {"load_in_4bit": True, "device_map": "auto", "bnb_4bit_quant_type": "fp4"}},
    ]

# Adjust configurations if running on CPU (bitsandbytes quantization typically requires CUDA)
if device.type == "cpu":
    print("Bitsandbytes quantization (INT8, NF4, FP4) usually requires CUDA. Filtering configurations.")
    configurations = [config for config in configurations if "bitsandbytes" not in config["name"]]

print(f"\n--- Starting Experiment for Model: {MODEL_NAME} ---")

for config in configurations:
    print(f"\nLoading model with configuration: {config['name']}")
    model = None # Ensure model is reset
    try:
        ### TODO: Load the model using AutoModelForCausalLM.from_pretrained()
        # Use the load_args from the current 'config'.
        # Handle device placement correctly (model.to(device) if not using device_map - e.g. in CPU usecase).
        # Example for handling device_map on CPU (though bitsandbytes won't quantize):
        current_load_args = config['load_args']
        # device_map is unnecessary on CPU
        if device.type == "cpu":
            current_load_args.pop("device_map", None)

        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            **current_load_args
        )

        # device_map already handles placement when present
        if "device_map" not in current_load_args:
            model = model.to(device)

        if model is None: # Check if model loading was skipped/failed in TODO
            print(f"Skipping {config['name']} due to model loading not implemented in TODO.")
            continue

        ### TODO: Get the model memory footprint using get_model_memory_footprint
        memory_mb = get_model_memory_footprint(model) # Placeholder
        print(f"Memory Footprint: {memory_mb:.2f} MB")

        avg_latencies_for_config = []
        generated_outputs_for_prompts = {}

        # Warm-up run
        _ = generate_text_and_time(
            model,
            tokenizer,
            PROMPTS[0],
            MAX_NEW_TOKENS
        )

        for i, prompt_text in enumerate(PROMPTS):
            print(f"  Processing prompt: '{prompt_text[:30]}...' ")
            prompt_specific_latencies = []
            current_generated_text = "N/A"
            
            ### TODO: Implement the timing loop (NUM_TIMING_RUNS)
            # Only perform multiple timing runs for the first prompt to establish 'Avg Latency (s)'
            # For other prompts, generate text once for quality assessment.
            # Store the first generation's text in 'current_generated_text'.
            # Accumulate latencies for the first prompt in 'prompt_specific_latencies'. Use generate_text_and_time function.
            # Collect the generated text for each prompt in generated_outputs_for_prompts
            if i == 0:
                for _ in range(NUM_TIMING_RUNS):
                    avg_latencies_for_config.append(generate_text_and_time(model, tokenizer, prompt_text, MAX_NEW_TOKENS)[1])
            elif i == 1:
                current_generated_text = generate_text_and_time(model, tokenizer, prompt_text, MAX_NEW_TOKENS)[0]
                generated_outputs_for_prompts[prompt_text] = current_generated_text
            else:
                generated_outputs_for_prompts[prompt_text] = generate_text_and_time(model, tokenizer, prompt_text, MAX_NEW_TOKENS)[0]


        overall_avg_latency_for_config = sum(avg_latencies_for_config) / len(avg_latencies_for_config) if avg_latencies_for_config else float('nan')

        results_list.append({
            "Float Precision": config["name"],
            "Memory (MB)": memory_mb,
            "Avg Latency (s)": overall_avg_latency_for_config, # Based on first prompt's timing
            **generated_outputs_for_prompts
        })
        
        del model # Free up memory
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    except Exception as e:
        print(f"Could not run configuration {config['name']}. Error: {e}")
        results_list.append({
            "Float Precision": config["name"],
            "Memory (MB)": "Error",
            "Avg Latency (s)": "Error",
            **{f"Prompt {i+1} Output": "Error" for i in range(len(PROMPTS))}
        })

# --- Display Results ---
df_results = pd.DataFrame(results_list)
print("\n\n--- Experiment Results Summary ---")
print(df_results.to_string())

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(



--- Starting Experiment for Model: gpt2 ---

Loading model with configuration: FP16 (Baseline GPU)


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Memory Footprint: 249.35 MB


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


...

  Processing prompt: 'The capital of France is...' 


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


  Processing prompt: 'Once upon a time, in a land fa...' 


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


  Processing prompt: 'To be or not to be, that is th...' 


--- Experiment Results Summary ---
        Float Precision  Memory (MB)  Avg Latency (s)                                                                                                                                                                                                                Once upon a time, in a land far, far away,                                                                                                                                                                                          To be or not to be, that is the
0   FP16 (Baseline GPU)   249.350121         0.504714          Once upon a time, in a land far, far away, the world was a land of the dead, and the dead were the living.\n\nThe dead were the living, and the living were the living.\n\nThe dead were the living, and the living were the living.\n\nThe dead  To be or not to be, that is the question.\n\nThe question is, what is the diffe

## 5. Analysis and Discussion

Based on the 'Experiment Results Summary' table printed above, analyze your findings:

1.  **Memory Scaling:** 
    *   **TODO**: Describe how the model's memory footprint scaled as you reduced precision. Quantify the reductions.
    *   **Answer**: the memory got reduced by each quantization we did, the modt reduction we got is from FP16 to INT8
        * FP16 - INT8 = 81 MB
        * INT8 - NF4 = 40.5 MB
        * NF4 - FP4 = 0 MB

2.  **Latency Changes:** 
    *   **TODO**: Analyze the changes in generation latency. Did latency always decrease with lower precision, or were there other factors at play? Explain potential reasons.
    *   **Answer**:
        * Not really, INT8 quantization is slower than FP16, that's probably duo to GPU being able to run FP16 operations faster than INT8
        * else, each quantization reduced the latency

3.  **Output Quality Degradation:** 
    *   **TODO**: Based on your subjective review of the outputs for each prompt and precision, at what point (if any) did you start to observe significant degradation in the output quality (e.g., coherence, relevance, repetitiveness)? Provide examples.
    *   **Answer**: for each quantization step we made, we lost some of the quality.
        * at INT8: the quality degraded at longer context, and we got repeating text such as "the dead were the living," and "\n\nI am not saying that I am a fan of the game, but I am saying that I am a fan of the game."
        * at NF4 and FP4: we got the same degradation level, on shorter sequwnces, text is getting repeated such as " there is a man who has been given a name, " and "\n\nThe only way to be or not to be is to be or not to be"

4.  **Key Trade-offs:** 
    *   **TODO**: Conclude with a summary of the key trade-offs you observed between memory savings, inference speed, and text quality when applying different quantization levels to GPT-2. Which configuration seemed to offer the best balance for which scenario?
    *   **Answer**: The best balance is INT8, it degraded the qualuty, but at longer sequences, so over all we notice improvements in performance and reduction in quality of the responses, so it depends on the user's requirements to decide what level of quality and performance needed to balance it out (regardless of the latency, as it's a bit slower)